In [2]:
import pandas as pd
import os.path as op
import os
import pandas as pd
from os import listdir
import numpy as np

bids_folder_local = '/Users/mrenke/data/ds-stressrisk'
bids_folder_ex = '/Volumes/mrenkeED/data/ds-stressrisk'

bids_folder = bids_folder_ex
subFolders = [f for f in listdir(bids_folder) if f[0:3] == 'sub']

In [5]:
import arviz as az

idata_NLC = az.from_netcdf(op.join(bids_folder_ex,'derivatives/cogmodels/model-NLC_1_trace.netcdf'))

n1h_df = idata_NLC.posterior['n1_hat'].to_dataframe().groupby('n1_hat_dim_0').mean() # average over trials
n2h_df = idata_NLC.posterior['n2_hat'].to_dataframe().groupby('n2_hat_dim_0').mean() # average over trials


In [6]:
from stress_risk.behavior.utils import get_data

df = get_data(bids_folder_local)
df = df.reset_index()
df = df.set_index(['subject','session','trial_nr'])

df['Vhat1'] = df['prob1'] * np.exp(n1h_df.values.flatten()) # changed here to exp. 
df['Vhat2'] = df['prob2'] * np.exp(n2h_df.values.flatten())

/Users/mrenke/mambaforge/envs/behav_fit2/lib/python3.10/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '
/Users/mrenke/mambaforge/envs/behav_fit2/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
for sub in subFolders:

    for ses in [1,2]:

        df_fil = df.xs(1,0,'subject').xs(1,0,'session')

        for run in range(1,7):
            df_timings = pd.read_table(op.join(bids_folder, sub,f'ses-{ses}','func', f'{sub}_ses-{ses}_task-risk_run-{run}_events.tsv'))

            for i in range(0,len(df_timings)):
                trial_nr = df_timings.loc[i]['trial_nr']
                
                df_timings.loc[i,'n1'] = df_fil.loc[trial_nr,'Vhat1']
                df_timings.loc[i,'n2'] = df_fil.loc[trial_nr,'Vhat2']

            fn = op.join(bids_folder, sub,f'ses-{ses}','func', f'{sub}_ses-{ses}_task-risk_run-{run}_value_events.tsv')
            df_timings.to_csv(fn, index=False, sep='\t')
